# Reproducing Wirth et al. (2017) from CRCNS hc-12

Paper. Wirth S, Baraduc P, Plante A, Pinede S, Duhamel JR (2017). Gaze-informed, task-situated
representation of space in primate hippocampus during virtual navigation. PLoS Biology 15(2):
e2001045. [doi:10.1371/journal.pbio.2001045](https://doi.org/10.1371/journal.pbio.2001045).
The claims file (`claims/wirth-2017.yaml`) was written from the version of record (PMC5328243),
Results, Methods and the legends of Figs 1 to 7 and S1 to S7; S1 Appendix was not available.

Datasets. CRCNS [hc-12](https://doi.org/10.6080/K0R49NQV) (189 cells, the new-maze sessions;
every claim rests on it) and CRCNS [hc-21](https://doi.org/10.6080/K0GX48R9) (101 cells, a
familiar and a new maze per file), streamed from the public bucket `crcnsarchive`
(`hc-12/live/data.tar.gz`, `hc-21/live/data.tar.gz`). CRCNS data sets are not versioned; the
version is recorded as `unversioned`, and the MD5 sums from each data set's `checksums.md5`
for the files read are: hc-12 `data.tar.gz` 426c7d00c0358f2cc9873e0dbb331e2f (descriptor
version 0.7, 17 Jan 2017); hc-21 `data.tar.gz` dee5e8aa4a7b3c44b61b8cecfe622a0a (descriptor
version 0.52, 21 Dec 2018). Each data set has one published state (`live/`, with only
documentation under `orig/`), so the version pinned is the only one there is, and hc-12 is the
deposit the paper cites. hc-21 is read only to check whether it holds recordings that are also
in hc-12 and whether it holds the probe sessions hc-12 lacks; no claim rests on it.

| Claim | Tested here |
|---|---|
| gaze-anticipates-action, gaze-anticipates-landmarks | tested (eye-signal sign assumed) |
| learning-trials, learning-to-90 | tested |
| probe-transfer | tested on monkey K; monkey S has no probe session in hc-12 |
| probe-faster-criterion | blocked: methods underspecified (learning-curve method of reference 60) |
| cells-recorded | tested |
| responsive-cells, multi-space-coding, state-space-ic-higher, sparsity-gaze-vs-state, spatial-selectivity-fraction | blocked: the point-of-gaze space cannot be built (no eye calibration); three-space results shown |
| state-space-most-responsive | tested on the state-space count, which needs no gaze |
| direction-center-vs-periphery, direction-tuning-not-maintained | tested |
| sparsity-state-vs-direction, sparsity-state-vs-position | tested |
| state-space-selectivity, landmark-identity-distance, landmark-identity-center | tested by bounds on the responsive set |
| path-selectivity, gaze-aligned-dynamics, latency-skew, eccentricity-modulation, foveation-selectivity, gaze-map-landmarks | blocked: need calibrated gaze |
| fields-per-cell, dimensionality-control | blocked: methods underspecified (S1 Appendix) |

In [ ]:
import resource
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import wirth

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

## One pass over hc-12

The archive is streamed once. For each cell file the helper keeps the new-maze trials, the
camera rows inside the new-maze session, the spikes and the eye signal, computes every map
measure with 999 surrogates under both surrogate readings, the landmark ANOVAs, and, once per
session, the behaviour. Seeds are fixed per cell (CRC32 of the file name).

In [ ]:
t0 = time.time()
metas, cells, beh = wirth.collect("hc-12", log=None)
M = pd.DataFrame(metas)
C_all = pd.DataFrame(cells)
print(f"hc-12 pass: {time.time() - t0:.0f} s, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

In [ ]:
t0 = time.time()
fp21 = pd.DataFrame(wirth.fingerprint("hc-21", log=None))
print(f"hc-21 pass: {time.time() - t0:.0f} s, {len(fp21)} files")

## Data check

In [ ]:
print("Files by monkey (ke = monkey K, sn = monkey S):")
print(M.groupby("monkey").agg(files=("file", "size"), with_new_maze=("has_newallo", "sum"),
                              sessions=("session", "nunique")).to_string())
print("\nFiles without a new-maze (newallo) field:", M.loc[~M.has_newallo, "file"].tolist())
print("\nSpike times taken from:", M.spike_src.value_counts().to_dict())
print("Top-level field sets:", M.fields.nunique(), "different layouts")
print("Camera frame interval (ms):", M.frame_dt_ms.describe()[["min", "max"]].round(3).to_dict())
print("Eye sample interval (ms):", M.eye_dt_ms.describe()[["min", "max"]].round(3).to_dict(),
      "; files without eye data:", int((~M.has_eye.fillna(False).astype(bool)).sum() - (~M.has_newallo).sum()))
print("Spike times sorted in every file:", bool(M.spikes_sorted.dropna().all()))
rng_cols = ["x_min", "x_max", "y_min", "y_max", "theta_min", "theta_max", "abscissa_max"]
print("\nRanges of the camera variables over files (min / max of each):")
print(M[rng_cols].agg(["min", "max"]).round(2).to_string())
print("\nTrials per new-maze session (per file):", M.n_trials.describe()[["min", "50%", "max"]].to_dict())
print("Session length, first trial start to last trial end (min):",
      ((M.t_end - M.t_start) / 60).describe()[["min", "50%", "max"]].round(1).to_dict())

The descriptor (hc-12, version 0.7) gives the positions as virtual X, Y. The values span x from
-16 to 12.9 and y from -15.2 to 15.2, five arm ends at radius 16 m (Methods: radius 16 m), so
positions are in metres. The camera orientation is in degrees, on 0 to 360 in some files and
-180 to 180 in others; the analysis takes it modulo 360. Column 5 (curvilinear abscissa along
the branch) runs from 0 at the centre to 1 at the arm end, a fraction of the arm, not metres.
Camera rows are at 60 Hz and eye samples at 200 Hz (5 ms), where the descriptor says 250 Hz.

In [ ]:
B = pd.DataFrame([{k: v for k, v in b.items() if k in ("monkey", "session", "n_trials", "learningtrial", "has_eye", "eye_scale", "eye_saturated")}
                  for b in beh.values()])
lc_ok = M.lc_order_ok.dropna()
print("Learning curves: column 2 <= column 1 <= column 3 in", int(lc_ok.sum()), "of", len(lc_ok), "files")
crit_files = []
for b in beh.values():
    crit = wirth.lower_bound_criterion(b["lc"][:, 1])
    crit_files.append((b["learningtrial"], crit))
same = sum((np.isnan(c) and lt == -2) or lt == c for lt, c in crit_files)
print("Field learningtrial equals the first trial with column 2 > 1/4 (or -2 when never):", same, "of", len(crit_files), "sessions")
print("\nEye signal scale by session:", B.eye_scale.value_counts(dropna=False).to_dict())
print("Fraction of eye samples at the +-5000 / +-5 rail, by session: median",
      round(B.eye_saturated.median(), 4), "max", round(B.eye_saturated.max(), 3))
rew = pd.Series({s: tuple(np.unique(b["chosen_arm"][b["correct"]]).astype(int)) for s, b in beh.items()})
print("\nArm chosen on rewarded trials (1-5 = codes 50-54), sessions per value:", rew.value_counts().to_dict())
print("\nCamera abscissa (0 centre, 1 arm end) at event times, median over sessions:")
ab = pd.DataFrame([{f"{k[0]} {k[1]:+.0f}s": v for k, v in b["abscissa_at_events"].items()} for b in beh.values()])
print(ab.median().round(2).to_string())

Checks against the description:
- The descriptor says the learning curve's middle column is the probability of a correct
  response, with the bounds in columns 1 and 3. In every file column 2 is the lowest and column
  3 the highest, so column 1 is the probability, column 2 the lower bound and column 3 the upper
  bound. The field `learningtrial`, which the descriptor does not describe, equals the first
  trial whose column 2 exceeds 1/4 (S3B legend's criterion), which confirms that reading.
- The descriptor says code 43 is the rewarded start arm and code 53 the rewarded chosen arm. The
  arm chosen on rewarded trials differs between sessions (printed above), as each day had a new
  reward location (Results); 53 is the rewarded arm only in some sessions.
- The eye signal is in undocumented units, on a +-5000 scale in some sessions and a +-5 scale in
  others, with a rail at the bound; no calibration to degrees is given. Eye samples are 5 ms
  apart (200 Hz), where the descriptor says 250 Hz.
- The camera rows share the trial clock: at the trial start (code 21) the abscissa is at the arm
  end, at the arrival at the choice point (code 22) it is 0, and one second later or earlier it is
  not (printed above).

In [ ]:
C = C_all[C_all.n_spikes_window > 100].copy()
print("Cells with a new maze:", len(C_all), "; with more than 100 spikes in the new-maze session:", len(C),
      C.groupby("monkey").size().to_dict())
sh_cols = [c for c in C.columns if c.startswith("shift_ic|")]
resp_state = C["icp|('full', 'state')|permute"] < 0.01
rel = C.loc[resp_state, sh_cols].div(C.loc[resp_state, "shift_ic|0.0"], axis=0)
print("\nClock check for the spikes: state-space IC with the spikes shifted in time, relative to no shift,")
print("median over the", int(resp_state.sum()), "cells with significant state-space IC (and over all cells, unnormalized):")
print(pd.DataFrame({"relative (sig. cells)": rel.median().round(3).values,
                    "IC bits/spike (all cells)": C[sh_cols].median().round(3).values},
                   index=[c.split("|")[1] + " s" for c in sh_cols]).T.to_string())

The spike clock check selects cells at zero shift, so the peak there is partly a selection
effect; the fall of the information with shifts of 2 to 5 s is consistent with spikes and camera
sharing one clock, without proving it. The eye clock is checked with the turn codes in the
section on gaze anticipation.

In [ ]:
# Recordings deposited twice: hc-12 against hc-21, by spike count and first spike times.
fp12 = M[M.has_newallo][["file", "session", "n_spikes_file", "first_spikes"]].copy()
fp12["head"] = fp12.first_spikes.apply(lambda t: tuple(np.round(t[:5], 3)))
fp21m = fp21[fp21.has_maze].copy()
fp21m["head"] = fp21m.spikes_head.apply(lambda t: tuple(np.round(t[:5], 3)))
pairs = fp12.merge(fp21m, on="head", suffixes=("_hc12", "_hc21"))
print("hc-21 files:", len(fp21), "; with a new maze:", int(fp21.has_maze.sum()),
      "; sessions:", fp21.session.nunique(), fp21.drop_duplicates("session").session.str[:2].value_counts().to_dict())
print("hc-12 / hc-21 file pairs with the same first five spike times:", len(pairs))
if len(pairs):
    print(pairs[["file_hc12", "file_hc21", "n_spikes_file", "n_spikes", "n_trials"]].to_string())
print("Sessions (dates) in both data sets:", sorted(set(M.session) & set(fp21.session)))
dup12 = fp12.groupby(["head", "n_spikes_file"]).file.apply(list)
print("hc-12 files sharing spike trains with another hc-12 file:", [v for v in dup12 if len(v) > 1])

## cells-recorded

> Of 270 cells recorded in the right hippocampus (128 in monkey S, 142 in monkey K), 189 fired
> more than 100 spikes per session and were analysed.

In [ ]:
cr = pd.DataFrame({"files": M.groupby("monkey").size(),
                   "with_new_maze": M.groupby("monkey").has_newallo.sum(),
                   "over_100_spikes": C.groupby("monkey").size()})
cr.loc["total"] = cr.sum()
print(cr.to_string())
print("\nRecording locations (AP, ML, DV relative to the interaural line) present in", int(M.reclocation.notna().sum()), "files;",
      "the files give no subfield (CA3, CA1, dentate gyrus).")

**Result: partially reproduced.** hc-12 holds 189 cell files (99 of monkey K, 90 of monkey S),
the paper's analysed count; 187 have new-maze data and 186 of them fire more than 100 spikes in
the new-maze session (expected 189 +- 18.9). The 270 cells recorded (128 S, 142 K) are not all
deposited: only the 189 are, so that count cannot be checked here.

## Learning: learning-trials and learning-to-90

> Both monkeys learned the rewarded arm in about a dozen trials; monkey S performed above chance
> after 12.3 plus or minus 2 trials (SEM) and monkey K after 14.8 plus or minus 2.4.

The curves are the files' own (columns as established in the data check). Probe sessions are
defined in the probe-transfer section; reading (b) leaves them out. Sessions whose lower bound
never exceeds 1/4 have no trial to criterion and are counted apart.

In [ ]:
def probe_info(b):
    sa = b["start_arm"]
    first = set(sa[:10][np.isfinite(sa[:10])].astype(int))
    if len(first) > 2:
        return False, None, first
    intro = next((i for i in range(len(sa)) if np.isfinite(sa[i]) and int(sa[i]) not in first), None)
    return intro is not None, intro, first


rows = []
for s, b in beh.items():
    lc = b["lc"]
    crit = wirth.lower_bound_criterion(lc[:, 1])
    up = [u for u in np.nonzero(lc[:, 2] >= 0.9)[0] if np.isfinite(crit) and u + 1 >= crit]
    is_probe, intro, first = probe_info(b)
    rows.append({"session": s, "monkey": b["monkey"], "n_trials": b["n_trials"], "probe": is_probe,
                 "intro": intro, "first_arms": sorted(first), "crit": crit,
                 "t90": (up[0] + 1 - crit) if up else np.nan, "pc": b["correct"].mean()})
S = pd.DataFrame(rows)

In [ ]:
for name, sub in (("(a) all sessions", S), ("(b) sessions that are not probe sessions", S[~S.probe])):
    g = sub.groupby("monkey").crit.agg(mean="mean", sem="sem", sessions_reaching="count",
                                       sessions_never=lambda x: int(x.isna().sum()), median="median")
    print(name); print(g.round(2).to_string()); print()
print("Expected: monkey S (sn) 12.3 +- 2, monkey K (ke) 14.8 +- 2.4")

**Result: reproduced.** Under reading (a) monkey S reaches criterion after 12.26 +- 3.28 trials
(mean +- SEM, 23 sessions) and monkey K after 16.53 +- 2.69 (38 sessions), both within the
paper's 12.3 +- 2 and 14.8 +- 2.4. Without the probe sessions (b) monkey K gives 15.82 +- 3.44,
also within. Two sessions of S and three of K never reach the criterion and are left out. The
SEMs here are larger than the paper's.

In [ ]:
for name, sub in (("(a) all sessions", S), ("(b) not probe sessions", S[~S.probe])):
    reach = sub.t90.notna()
    print(f"{name}: sessions reaching both criteria {int(reach.sum())} of {len(sub)}; "
          f"fewer than 10 trials from lower bound > 1/4 to upper bound >= 0.9: {int((sub.t90 < 10).sum())}, "
          f"fraction of sessions reaching {(sub.t90 < 10).sum() / reach.sum():.3f}, of all sessions {(sub.t90 < 10).mean():.3f}")
print("Trials from criterion to upper bound 0.9 (sessions reaching):", S.t90.describe()[["min", "50%", "max"]].to_dict())
print("Expected 0.89 +- 0.089")

**Result: reproduced.** Of the 59 sessions that reach both criteria, 53 need fewer than 10
trials (0.898, expected 0.89 +- 0.089); counted over all 66 sessions the fraction is 0.803, also
inside the tolerance (0.801 to 0.979). Without the probe sessions the fractions are 0.898 and
0.815.

## probe-transfer

> In probe sessions, performance after the new western entries were introduced was higher than
> at the start of the session (monkey K 73% against 44%, monkey S 80% against 55%).

A probe session is read from the start arms: the first 10 trials use one or two entries and
later trials use others (Results; S3C legend: one entry for 36 trials, or two entries until trial
60). The introduction is the first trial from an entry not used in the first 10.

In [ ]:
P = S[S.probe].copy()
P["pc_first5"] = [beh[s]["correct"][:5].mean() for s in P.session]
P["pc_after5"] = [beh[s]["correct"][int(i):int(i) + 5].mean() for s, i in zip(P.session, P.intro)]
print(P[["session", "monkey", "n_trials", "first_arms", "intro", "pc_first5", "pc_after5"]].to_string())
print("\nProbe sessions per monkey:", P.groupby("monkey").size().to_dict(), "(paper: 9 for S, 15 for K)")
print("Sessions of monkey S whose first 10 trials use at most two entries:",
      int(((S.monkey == "sn") & S.first_arms.apply(len).le(2)).sum()))
print((P.groupby("monkey")[["pc_first5", "pc_after5"]].mean() * 100).round(1).to_string())
if len(P) > 1:
    w = stats.wilcoxon(P.pc_after5, P.pc_first5)
    print(f"Wilcoxon signed-rank, after against first 5 trials, {len(P)} sessions: W = {w.statistic}, p = {w.pvalue:.4f}")
    print("Median difference (after - first):", (P.pc_after5 - P.pc_first5).median())
print("\nExpected: K 73 +- 7.3 after, 44 +- 4.4 at the start; S 80 +- 8, 55 +- 5.5")
# hc-21: sessions whose first 10 start codes use at most two entries
fp21m["first_arms"] = fp21m.codes_head.apply(lambda c: sorted(set(c)))
p21 = fp21m[fp21m.first_arms.apply(len) <= 2].drop_duplicates("session")
print("hc-21 new-maze sessions whose first 10 trials use at most two entries:", p21.session.tolist())

**Result: not reproduced.** hc-12 holds 12 probe sessions of monkey K (the paper has 15) and
none of monkey S (the paper has 9); hc-21 adds no other. For K, performance is 28.3% over the
first 5 trials and 58.3% over the 5 trials after the new entries, both below the paper's 44% and
73% by more than the tolerance, and the Wilcoxon signed-rank test gives p = 0.072. The direction
of the effect is the paper's (median gain 0.30).

## gaze-anticipates-action

> During the 500 ms before each joystick action, gaze anticipated the direction of the
> subsequent movement.

Each left (code 2) or right (code 1) joystick move; mean horizontal eye signal in the 500 ms
(reading a) or 300 ms (reading b) before it, for successful trials (Method W) and, as a variant,
all trials. The eye signal is in uncalibrated units on two scales (data check), so values are
standardized within each session (minus the session median, over the session's SD) before
pooling sessions; per-session differences are also counted. Wilcoxon rank-sum, right against
left, per monkey (reading i) and pooled (reading ii).

In [ ]:
T = pd.DataFrame([dict(session=s, monkey=b["monkey"], **t) for s, b in beh.items() for t in b["turns"]])
print("Orientation change in the 600 ms after the code, median (deg):", T.groupby("code").dtheta_0_600ms.median().round(1).to_dict(),
      "(code 1 = right lowers the camera orientation)")


def zsess(df, col):
    g = df.groupby("session")[col]
    return (df[col] - g.transform("median")) / g.transform("std")


def rank_test(df, col, right):
    x = df[col].dropna()
    a, b = x[right.loc[x.index]], x[~right.loc[x.index]]
    if len(a) < 2 or len(b) < 2:
        return np.nan, np.nan, len(a), len(b)
    return float(a.median() - b.median()), float(stats.mannwhitneyu(a, b).pvalue), len(a), len(b)

In [ ]:
res = []
for trials_used, Tsub in (("successful", T[T.correct]), ("all", T)):
    Tsub = Tsub.copy()
    for w in ("eye500", "eye300"):
        for eye in ("mean", "l", "r"):
            col = f"{w}_{eye}"
            Tsub["z"] = zsess(Tsub, col)
            for mk in ("ke", "sn", "both"):
                d = Tsub if mk == "both" else Tsub[Tsub.monkey == mk]
                diff, p, na, nb = rank_test(d, "z", d.code == 1)
                sess_diff = d.groupby("session").apply(lambda g: g[g.code == 1][col].median() - g[g.code == 2][col].median()).dropna()
                res.append({"trials": trials_used, "window": w[3:] + " ms", "eye": eye, "monkey": mk,
                            "n_right": na, "n_left": nb, "median z right-left": round(diff, 3), "p": p,
                            "sessions right<left": f"{int((sess_diff < 0).sum())}/{len(sess_diff)}"})
R1 = pd.DataFrame(res)
print(R1.to_string())
lags = [c for c in T.columns if c.startswith("lag")]
Tc = T[T.correct].copy()
prof = {}
for mk in ("ke", "sn"):
    d = Tc[Tc.monkey == mk].copy()
    for c in lags:
        d[c] = zsess(d, c)
    prof[mk] = (d[d.code == 1][lags].median() - d[d.code == 2][lags].median()).values
print("\nEye clock check: median standardized eye signal, right minus left turns, in 100 ms windows from the code:")
print(pd.DataFrame(prof, index=[f"{float(c[3:]):+.1f}" for c in lags]).T.round(2).to_string())

Sign of the eye signal. The files do not say which sign of the horizontal signal is rightward.
During a turn the view rotates, and the mean eye position in optokinetic stimulation shifts
toward the side of the quick phases, which is the side the observer turns to. The difference
right minus left grows in the same (negative) direction from the turn on, in both monkeys
(profile above). The analysis therefore takes a negative value as rightward, an assumption.

**Result: reproduced, resting on the assumed sign of the eye signal.** Before right turns the
horizontal eye signal lies on the negative side of that before left turns in each monkey, in
both windows, for each eye and both eyes, with rank-sum p below 1e-100 (500 ms, both eyes,
successful trials: K median difference -0.77 session SD, p = 4.0e-148; S -1.00, p = 4.3e-146),
and in 39 of 40 sessions of K and 20 of 20 of S. Taking negative as rightward (the profile
after the turn grows in the same direction), gaze is on the side of the coming turn.

## gaze-anticipates-landmarks

> When relocated to a new entry, the monkeys gazed proactively at the location where the
> landmarks would appear.

Passive returns (tags 5xy between the end of a trial and the next start); a return turns right
when the camera orientation falls over it (as for code 1). Mean horizontal eye signal over the
return, standardized within session; rank-sum right against left. Returns after all trials,
and after successful trials as a variant (Method W).

In [ ]:
Rt = pd.DataFrame([dict(session=s, monkey=b["monkey"], **r) for s, b in beh.items() for r in b["returns"]])
res = []
for trials_used, Rsub in (("all", Rt), ("after successful", Rt[Rt.correct])):
    Rsub = Rsub.copy()
    for eye in ("mean", "l", "r"):
        Rsub["z"] = zsess(Rsub, f"eye_{eye}")
        for mk in ("ke", "sn", "both"):
            d = Rsub if mk == "both" else Rsub[Rsub.monkey == mk]
            diff, p, na, nb = rank_test(d, "z", d.dtheta < 0)
            sd = d.groupby("session").apply(lambda g: g[g.dtheta < 0][f"eye_{eye}"].median() - g[g.dtheta >= 0][f"eye_{eye}"].median()).dropna()
            res.append({"returns": trials_used, "eye": eye, "monkey": mk, "n_right": na, "n_left": nb,
                        "median z right-left": round(diff, 3), "p": p, "sessions right<left": f"{int((sd < 0).sum())}/{len(sd)}"})
print("Median absolute orientation change over a return (deg):", round(Rt.dtheta.abs().median(), 1))
print(pd.DataFrame(res).to_string())

**Result: reproduced, resting on the assumed sign of the eye signal.** During passive returns
that turn right the eye signal lies on the negative (rightward, as assumed above) side of that
during returns that turn left, in each monkey (all returns, both eyes: K -0.56 session SD,
p = 6.2e-58, 39 of 41 sessions; S -0.56, p = 6.4e-30, 20 of 21 sessions), and after successful
trials only (p = 4.7e-34 and 1.5e-10).

## Information content in the coding spaces

The point-of-gaze space needs the gaze direction in degrees, which the files cannot give (the
eye signal is uncalibrated, data check). Position, direction and state space are computed for
the cells with more than 100 spikes in the new-maze session. Readings: trial window "full"
(start to next start) or "toreward" (start to reward); surrogates "permute" or "shift"
(Method W). Significance is the bilateral surrogate p < 0.01. The responsive set of the paper
needs all four spaces; the set responsive in the three spaces here is contained in it, and all
cells contain it, so counts over these two sets bound the count over the paper's set.

In [ ]:
READINGS = [("full", "permute"), ("full", "shift"), ("toreward", "permute"), ("toreward", "shift")]
SPACES = ("position", "direction", "state")


def sig(space, r, sr, alpha=0.01):
    return C[f"icp|('{r}', '{space}')|{sr}"] < alpha


rows = []
for r, sr in READINGS:
    sg = {k: sig(k, r, sr) for k in SPACES}
    n_sig = sum(v.astype(int) for v in sg.values())
    rows.append({"trials": r, "surrogates": sr, **{k: int(v.sum()) for k, v in sg.items()},
                 "any of three": int((n_sig > 0).sum()), "two or more": int((n_sig >= 2).sum()),
                 "cells": len(C)})
IC3 = pd.DataFrame(rows)
print(IC3.to_string())
print("\nMedian number of valid bins per map:", C[[c for c in C.columns if c.startswith("nbins|")]].median().to_dict())

## responsive-cells (blocked) and state-space-most-responsive

> 111 of the 189 cells (59%) had significant information content in at least one of the four
> coding spaces.

> The proportions of cells responsive to each space were unequal; the state space accounted for
> the most cells (84), and more cells were responsive to state space than to point of gaze.

In [ ]:
print("Cells with significant state-space IC (p < 0.01), per reading, and per monkey:")
for r, sr in READINGS:
    s_ = sig("state", r, sr)
    print(f"  {r}/{sr}: {int(s_.sum())} of {len(C)}", s_.groupby(C.monkey).sum().to_dict())
print("Expected 84 +- 8.4 (75.6 to 92.4)")
print("\nThree-space responsive cells per reading (a lower bound of the paper's 111 +- 11.1):", IC3["any of three"].tolist())
print("One-sided variant (upper tail p < 0.01, i.e. bilateral p < 0.02), state space:",
      [int(sig("state", r, sr, 0.02).sum()) for r, sr in READINGS])

**Result: not reproduced.** The state-space count is 57 of 186 cells under the main reading
(full trials, permuted chunks) and 33, 42 and 24 under the others, all below the expected
75.6 to 92.4. The count needs no gaze, so the missing gaze space does not change this; the
chi-squared tests across the four spaces could not be run. State space is still the space with
the most significant cells of the three computed here (57 against 46 and 40).

## direction-center-vs-periphery

> For direction-selective cells, both information per spike and sparsity of directional firing
> were higher in the maze center than on the peripheral return paths.

Directional maps from centre time (tags 2xy, 3xy) and return paths (tags 5xy) of successful
trials with their returns (reading "full"), at one common angular resolution per session.
Normalized IC and sparsity (minus the surrogate mean); Wilcoxon signed-rank across cells.

In [ ]:
dir_rows = []
for sr in ("permute", "shift"):
    for cellset, mask in (("(a) direction IC p < 0.01", sig("direction", "full", sr)),
                          ("(b) three-space responsive", sum(sig(k, "full", sr).astype(int) for k in SPACES) > 0)):
        d = C[mask]
        for meas in ("ic", "sp"):
            a, b_ = d[f"{meas}n|center|{sr}"], d[f"{meas}n|return|{sr}"]
            ok = a.notna() & b_.notna()
            w = stats.wilcoxon(a[ok], b_[ok])
            dir_rows.append({"surrogates": sr, "cells": cellset, "measure": {"ic": "IC", "sp": "sparsity"}[meas],
                             "n": int(ok.sum()), "median centre": round(a[ok].median(), 3),
                             "median return": round(b_[ok].median(), 3),
                             "median diff": round((a[ok] - b_[ok]).median(), 3),
                             "centre higher": int((a[ok] > b_[ok]).sum()), "p": w.pvalue})
print(pd.DataFrame(dir_rows).to_string())
print("\nValid direction bins: centre median", C["nbins|center"].median(), ", return median", C["nbins|return"].median())

**Result: not reproduced.** For the 40 direction-selective cells (reading a, permuted chunks)
normalized sparsity is higher in the centre (median difference 0.020, p = 0.031) but IC is not
(0.012, p = 0.43), so neither p < 0.001 nor both p < 0.05 holds. Over the 59 cells responsive in
the three spaces computed here, a stand-in for reading (b), both are higher with p = 0.045 and
0.0074, the criterion's partial level; with shifted chunks no comparison is significant.

## direction-tuning-not-maintained

> Direction tuning was not maintained across positions; the correlation between directional
> tuning in the center and on return paths was significantly low in 20 of 56 cells (36%), more
> than chance, and significantly high in only 3, consistent with chance.

Uncentered correlation over the bins valid in both maps (Methods, Directional correlations);
low or high when the actual value is in the lower or upper 2.5% of the surrogates; binomial
chance rate 0.025 (claims file). Cells with at least 10 common valid bins (a standard choice;
2 shown as a variant).

In [ ]:
rows = []
for sr in ("permute", "shift"):
    for cellset, mask in (("(a) direction IC p < 0.01", sig("direction", "full", sr)),
                          ("(b) three-space responsive", sum(sig(k, "full", sr).astype(int) for k in SPACES) > 0)):
        for min_common in (10, 2):
            d = C[mask & (C.n_common >= min_common) & C[f"corr_lo|{sr}"].notna()]
            lo = int((d[f"corr_lo|{sr}"] <= 0.025).sum())
            hi = int((d[f"corr_up|{sr}"] <= 0.025).sum())
            rows.append({"surrogates": sr, "cells": cellset, "min common bins": min_common, "n": len(d),
                         "low": lo, "high": hi, "p binom low": wirth.binom_upper(lo, len(d), 0.025),
                         "p binom high": wirth.binom_upper(hi, len(d), 0.025),
                         "median corr": round(d["corr"].median(), 3) if len(d) else np.nan})
print(pd.DataFrame(rows).to_string())
print("Expected: tested 56 +- 5.6, low 20 +- 2 with binomial p < 0.001, high 2 to 4")

**Result: partially reproduced.** Among the 40 direction-selective cells, 15 have a
significantly low centre-return correlation (38%, binomial p = 2.1e-14 at chance 0.025) and 4 a
significantly high one (p = 0.017). The proportion of low cells is the paper's (36%), and the
high count is inside 2 to 4, but 40 cells are tested (expected 56 +- 5.6) and the low count of
15 is outside 20 +- 2. Over the 59 three-space responsive cells, 16 are low and 4 high. The
minimum of common bins (10 or 2) changes nothing; with shifted chunks 5 of 24 are low.

## sparsity-state-vs-direction and sparsity-state-vs-position

> Sparsity was significantly higher in state space than in direction space.

> Sparsity was not significantly higher in state space than in position space overall, but the
> 15 cells whose difference was individually significant were sparser in state space.

Cells with significant IC in both spaces compared; per cell normalized state-space sparsity
minus the other space's; Wilcoxon signed-rank. The per-cell difference is significant when its
surrogate p (the difference recomputed on each surrogate) is below 0.05 (reading a) or 0.01 (b).

In [ ]:
def sparsity_compare(other):
    rows = []
    for r, sr in READINGS:
        both = sig("state", r, sr) & sig(other, r, sr)
        d = C[both]
        diff = d[f"spn|('{r}', 'state')|{sr}"] - d[f"spn|('{r}', '{other}')|{sr}"]
        w = stats.wilcoxon(diff) if len(diff) > 1 else None
        row = {"trials": r, "surrogates": sr, "cells in both": len(d), "median diff": round(diff.median(), 4),
               "state sparser": int((diff > 0).sum()), "p": w.pvalue if w else np.nan}
        for lvl, nm in ((0.05, "a"), (0.01, "b")):
            ind = d[f"spdiffp|{r}|{other}|{sr}"] < lvl
            sub = diff[ind]
            ws = stats.wilcoxon(sub) if len(sub) > 1 else None
            row.update({f"indiv sig ({nm})": int(ind.sum()), f"their median ({nm})": round(sub.median(), 4) if len(sub) else np.nan,
                        f"their p ({nm})": ws.pvalue if ws else np.nan})
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
print(sparsity_compare("direction").to_string())

**Result: reproduced under the main reading only.** For the 40 cells significant in both
state and direction space (full trials, permuted chunks), normalized sparsity is higher in state
space (median difference 0.016, 28 of 40 cells, Wilcoxon p = 0.0022). Under the other three
readings it is not significant (p = 0.32, 0.92, 0.36).

In [ ]:
print(sparsity_compare("position").to_string())
print("Expected: overall not significant (p >= 0.05); individually significant 15 +- 1.5, sparser in state space with p < 0.05")

**Result: not reproduced.** Under the main reading state space is significantly sparser than
position space overall (median difference 0.016, p = 0.00043), where the paper finds no
significant difference; 9 (level 0.05) or 5 (level 0.01) cells differ individually, against
15 +- 1.5. Under the readings where the overall test is not significant (shifted chunks), only
2 or 6 cells differ individually. No reading meets all three parts.

## state-space-selectivity

> In the maze center, at the same position and direction, 12 of 111 responsive cells (11%)
> showed significantly high state-space selectivity, discriminating the current rotational move,
> more than chance; only 4 showed significantly low selectivity, as expected by chance.

SSI per Methods, State-space selectivity; high or low when in the upper or lower 5% of the
surrogates (binomial chance 0.05, claims file). Counts over the three-space responsive cells
(lower bound) and over all cells (upper bound).

In [ ]:
rows = []
for r, sr in READINGS:
    resp3 = sum(sig(k, r, sr).astype(int) for k in SPACES) > 0
    for cellset, mask in (("three-space responsive", resp3), ("all cells", pd.Series(True, index=C.index))):
        d = C[mask & C[f"ssi_up|{sr}"].notna()]
        hi = int((d[f"ssi_up|{sr}"] <= 0.05).sum())
        lo = int((d[f"ssi_lo|{sr}"] <= 0.05).sum())
        rows.append({"trials (responsive set)": r, "surrogates": sr, "cells": cellset, "n": len(d), "high": hi, "low": lo,
                     "p binom high": wirth.binom_upper(hi, len(d), 0.05), "p binom low": wirth.binom_upper(lo, len(d), 0.05)})
print(pd.DataFrame(rows).to_string())
print("Cells without an SSI (fewer than one bin with 400 ms in two move types):", int(C.ssi.isna().sum()))
print("Bins entering the SSI, median:", C.ssi_bins.median())
print("Expected: high 12 +- 1.2 (11 to 13) with binomial p < 0.05; low 3 to 5, not significant")

**Result: not reproduced.** The paper's responsive set contains the three-space responsive
cells and is contained in all cells, so its counts lie between the two printed rows. Under the
main reading the high count lies between 15 and 23, above 11 to 13; with shifted chunks it lies
between 3 and 8, below. Under trials to reward with permuted chunks the high count (13 to 23)
could reach 13, but the low count lies between 7 and 12, above the 3 to 5 the criterion
allows. No reading can meet the criterion.

## landmark-identity-distance and landmark-identity-center

> A two-way ANOVA of firing on the entry paths by landmark viewed and relative distance gave a
> main effect of landmark identity in 30 of 111 responsive cells and of relative distance in 64;
> 29 cells coded a combination (14 with a significant interaction, 15 with both main effects),
> more than chance.

> The main effect of landmark identity was confirmed by a one-way ANOVA on firing to the four
> landmark views at the center, significant in 12 of 111 responsive cells, more than chance.

Which laps enter is read two ways. Method W takes successful trials only; the Methods' ANOVA
section says "firing rates were collected for each of the animal's laps on each of the four
entry paths", and Fig 5 reports F(9,756) for two cells recorded together. In hc-12 only one
session gives 756 residual degrees of freedom: sn20110624, with 193 trials, when every lap is
used (193 x 4 - 16 = 756; printed below). Both readings are run. Counts over the
three-space responsive cells and over all cells bound the count over the paper's responsive
set, as above.

In [ ]:
def lm_counts(d, laps):
    a = 0.05
    k2, k1 = ("lm2", "lm1") if laps == "successful" else ("lm2all", "lm1")
    lmk, rd, it = d[f"{k2}_p_lm"] < a, d[f"{k2}_p_rd"] < a, d[f"{k2}_p_int"] < a
    ca, cb = ("lm1a_p", "lm1b_p") if laps == "successful" else ("lm1aall_p", "lm1ball_p")
    return {"laps": laps, "n": len(d), "identity": int(lmk.sum()), "distance": int(rd.sum()), "interaction": int(it.sum()),
            "both, no interaction (a)": int((lmk & rd & ~it).sum()), "both (b)": int((lmk & rd).sum()),
            "centre one-way (a)": int((d[ca] < a).sum()), "centre one-way (b)": int((d[cb] < a).sum())}


rows = []
for laps in ("successful", "all"):
    for r, sr in READINGS:
        resp3 = sum(sig(k, r, sr).astype(int) for k in SPACES) > 0
        rows.append({"responsive set": f"three-space {r}/{sr}", **lm_counts(C[resp3], laps)})
    rows.append({"responsive set": "all cells", **lm_counts(C, laps)})
LM = pd.DataFrame(rows)
print("Entry paths per cell:", C.lm2_n_lm.value_counts().to_dict(), "; laps per cell, median: successful", C.n_laps.median(),
      ", all", C.n_laps_all.median())
print("Residual df of the two-way ANOVA, median: successful laps", C.lm2_df_res.median(), ", all laps", C.lm2all_df_res.median())
ex = C[C.session == "sn20110624"][["name", "n_trials", "lm2_df_res", "lm2all_df_res", "lm2all_F_int", "lm2all_p_int"]]
print("\nThe session with 193 trials, whose residual df with all laps is 193 x 4 - 16 = 756, as in Fig 5:")
print(ex.round(4).to_string())

In [ ]:
print(LM.drop(columns=["centre one-way (a)", "centre one-way (b)"]).to_string())
print("Expected: identity 30 +- 3, distance 64 +- 6.4, interaction 14 +- 1.4, both 15 +- 1.5")
fx = C[["name", "session", "lm2all_F_int", "lm2all_p_int", "lm2all_df_int", "lm2all_df_res"]].copy()
fx["dist"] = np.minimum(abs(fx.lm2all_F_int - 4.4076), abs(fx.lm2all_F_int - 2.1243))
print("\nCells whose interaction F (all laps) comes nearest the Fig 5 values (4.41 and 2.12):")
print(fx.nsmallest(4, "dist").round(4).to_string())

**Result: not reproduced.** With successful laps, at most 24 cells (all 186 cells) show a main
effect of relative distance, and at most 31 with all laps, against 64 +- 6.4; this bound holds
for any responsive set. The identity count lies between 16 and 32 (successful laps) or 24 and 47
(all laps), which allows 30, and the interaction count between 6 and 14 or 8 and 17, but the
distance count fails under every reading, and with it the partial level. All laps give the
paper's residual df of 756 in the one session of 193 trials; none of its three cells has the
Fig 5 F values (1.83, 3.19, 1.56 against 4.41 and 2.12).

In [ ]:
print(LM[["responsive set", "laps", "n", "centre one-way (a)", "centre one-way (b)"]].to_string())
print("Expected 12 +- 1.2 (11 to 13), binomial p < 0.05 at chance 0.05")
print("Groups in the one-way ANOVAs:", C.lm1a_groups.value_counts().to_dict(), C.lm1b_groups.value_counts().to_dict())

**Result: blocked (metadata insufficient).** Over the three-space responsive cells and over all
cells the one-way count lies between 9 and 16 (reading a) or 11 and 23 (reading b) with
successful laps, and between 8 and 18 or 12 and 26 with all laps, so it may or may not fall in
11 to 13, and the binomial test depends on the size of the responsive set. Without the
point-of-gaze space the paper's responsive set cannot be formed, and the claim cannot be
decided.

## Claims blocked by the missing gaze calibration: three-space results

These numbers do not decide an outcome; they show how far the claims can be followed without
the point-of-gaze space.

In [ ]:
r, sr = "full", "permute"
resp3 = sum(sig(k, r, sr).astype(int) for k in SPACES) > 0
print(f"Reading {r}/{sr}: responsive in three spaces {int(resp3.sum())} of {len(C)} ({resp3.mean():.2f});",
      f"two or more spaces {IC3.loc[0, 'two or more']}; position IC p < 0.01 {int(sig('position', r, sr).sum())} "
      f"({sig('position', r, sr).mean():.3f} of the cells)")
for other in ("position", "direction"):
    both = sig("state", r, sr) & sig(other, r, sr)
    d = C[both]
    diff = d[f"icn|('{r}', 'state')|{sr}"] - d[f"icn|('{r}', '{other}')|{sr}"]
    print(f"State-space minus {other} normalized IC, {len(d)} cells: median {diff.median():.3f}, Wilcoxon p = {stats.wilcoxon(diff).pvalue:.3g}")

## Summary

| Claim | Outcome |
|---|---|
| gaze-anticipates-action | reproduced (assumed sign of the eye signal) |
| gaze-anticipates-landmarks | reproduced (assumed sign of the eye signal) |
| learning-trials | reproduced |
| learning-to-90 | reproduced |
| probe-transfer | not reproduced (monkey K only; no probe session of S in hc-12) |
| probe-faster-criterion | blocked: methods underspecified |
| cells-recorded | partially reproduced (only the 189 analysed cells are deposited) |
| responsive-cells | blocked: metadata insufficient (no gaze calibration) |
| state-space-most-responsive | not reproduced |
| multi-space-coding | blocked: metadata insufficient |
| direction-center-vs-periphery | not reproduced |
| direction-tuning-not-maintained | partially reproduced |
| state-space-ic-higher | blocked: metadata insufficient |
| sparsity-state-vs-direction | reproduced (main reading only) |
| sparsity-state-vs-position | not reproduced |
| sparsity-gaze-vs-state | blocked: metadata insufficient |
| state-space-selectivity | not reproduced |
| landmark-identity-distance | not reproduced |
| landmark-identity-center | blocked: metadata insufficient |
| path-selectivity, gaze-aligned-dynamics, latency-skew, eccentricity-modulation, foveation-selectivity, gaze-map-landmarks | blocked: metadata insufficient |
| fields-per-cell, dimensionality-control | blocked: methods underspecified |
| spatial-selectivity-fraction | blocked: metadata insufficient |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The point-of-gaze space is missing, as the eye
signal has no calibration, so the responsive set of the paper cannot be formed; counts over it
are bounded by counts over the three-space responsive cells and over all cells. The map
resolution steps, the surrogate shifting (two readings), the trial windows (two readings), the
move types of the state-space selectivity, the per-session standardization of the eye signal and
its sign are choices the paper does not fix. The learning-curve method of the paper's reference
60 is not known, so curves were not recomputed.

What was found about the data. The learning curves' columns are probability, lower and upper
bound, not the order the descriptor gives; eye samples are 200 Hz, not 250 Hz; the eye signal
is on two different scales with no units or calibration; the rewarded arm is not always code 53;
two hc-12 files have no new-maze data; 87 of the 101 hc-21 files hold the same spike trains as
hc-12 files; hc-12 holds 12 of monkey K's 15 probe sessions and none of monkey S's 9.